# Bay Area Microclimate Weather Prediction

**Goal:** Build an ML model that predicts localized weather conditions across distinct microclimate zones in the San Francisco Bay Area.

**Approach:** Phase 1 GBT baseline (XGBoost/LightGBM) with terrain features → Phase 2 LSTM for temporal structure → predict at a regular lat/lon grid and cluster on model-predicted weather profiles to define microclimate zones.

**S3 Bucket:** `s3://bay-area-microclimate/`

---

## Project Architecture

```
┌─────────────────────────────────────────────────────────────────┐
│                        DATA SOURCES                             │
├───────────────────────────────┬─────────────────────────────────┤
│  Synoptic API                 │  ERA5 (via Open-Meteo)          │
│  ~600+ professional stations  │  42 grid pts (0.25°)            │
│  1-yr (primary training data, │  10-yr (synoptic context)       │
│  real observations)           │  boundary layer height,         │
│                               │  850hPa temp, etc.              │
└──────────────┬────────────────┴────────────────┬───────────────┘
               │                                 │
               ▼                                 ▼
┌─────────────────────────────────────────────────────────────────┐
│                     S3: raw/ (parquet)                           │
│  synoptic/monthly/YYYY-MM/chunk_XXXX.parquet                    │
│  era5/monthly/YYYY-MM/grid_{lat}_{lon}.parquet                  │
└──────────────────────────┬──────────────────────────────────────┘
                           │
                           ▼
┌─────────────────────────────────────────────────────────────────┐
│                     ML PIPELINE                                 │
│  Phase 1: GBT baseline (XGBoost/LightGBM)                      │
│     - Separate models: temp, humidity, fog/cloud, wind          │
│     - Inputs: observations + ERA5 + terrain features            │
│  Phase 2: LSTM wrapper for temporal patterns                    │
│     - Diurnal cycles (temp/humidity), fog persistence           │
│  Phase 3: Predict at regular lat/lon grid, cluster into zones   │
└──────────────────────────┬──────────────────────────────────────┘
                           │
                           ▼
┌─────────────────────────────────────────────────────────────────┐
│                 S3: features/ (parquet)                          │
│  static/stations_with_features.parquet                          │
│  zones/zone_assignments.parquet  (model-derived)                │
│  zones/zone_profiles.parquet     (model-derived)                │
└─────────────────────────────────────────────────────────────────┘
```

## Pipeline Status Dashboard

| Component | Script | Status | Notes |
|-----------|--------|--------|-------|
| Synoptic download | `src/download_synoptic.py` | **Partial** (234/299 chunks) | 18/23 chunks per month x 13 months. ~604 active stations. |
| ERA5 download | `src/download_era5.py` | **Complete** | 546 files (42 grid points x 13 months) |
| DEM terrain features | -- | Not collected | SRTM 30m or USGS 3DEP. Stage 1 blocker. |
| NLCD land cover | -- | Not collected | Stage 1 blocker. |
| ERA5 to station interpolation | -- | Not implemented | Bilinear interp to station lat/lons. Stage 1 blocker. |
| Static features | `src/compute_static_features.py` | Partial (in S3) | Has dist_coast, dist_bay. Missing DEM/NLCD. |
| Stage 1: GBT per variable | -- | Not implemented | XGBoost/LightGBM, one model per target variable |
| Stage 2: LSTM temp/humidity | -- | Not implemented | 24hr rolling window, sequential memory |
| Stage 3: Prediction grid + zones | -- | Not implemented | Predict at arbitrary lat/lon grid, cluster into microclimate zones |

*Geo-LSTM-Kriging rejected (Warsaw RMSE 3.0C, R2 0.58; missing vertical dimension for SF fog).*
*Weather Underground abandoned — unreliable API access model.*
*Open-Meteo dense grid removed — model-interpolated data, not real observations. Training on it would replicate another model's assumptions.*

---

## Data Sources Detail

### 1. Synoptic API (`src/download_synoptic.py`)

Surface weather observations from professional and amateur stations.

- **Stations:** ~1,124 in Bay Area bbox + 19 known ASOS stations (KSFO, KOAK, KSJC, etc.)
- **Variables:** `temp_c`, `humidity`, `wind_speed_kph`, `wind_dir_deg`, `precip_mm`
- **Chunking:** 50 stations per API call (~180k rows/month/chunk)
- **S3 layout:** `raw/synoptic/monthly/YYYY-MM/chunk_XXXX.parquet`
- **Full run:** 13 months x 23 chunks = 299 API calls
- **Free tier limit:** 1 year of history (hard cap — blocks the 20-year goal)

**Verified results (first run):**
| Chunk | Rows |
|-------|------|
| `chunk_0000` | 155,218 |
| `chunk_0001` | 182,085 |
| `chunk_0002` | 186,401 |

**Key gotchas:**
- Use `SYNOPTIC_TOKEN` (generated token), not the API key directly — API key returns 401
- Variable name is `precip_accum`, NOT `precip_accumulated` (wrong name returns 0 stations)
- No client-side RESTRICTED filter needed — API returns only accessible data

### 2. ERA5 Reanalysis (`src/download_era5.py`)

Large-scale atmospheric context via Open-Meteo Archive API.

- **Grid:** 6x6 = 36 points at 0.25° resolution (~27km)
- **History:** Back to 1940 (no free-tier limit). Downloading 10 years (2016–2026) for ENSO coverage.
- **Variables:**
  - Temperature (2m surface + 850hPa free-atmosphere)
  - Relative humidity, surface pressure, wind speed/direction
  - Precipitation, cloud cover
  - **Boundary layer height** (key for marine layer modeling)
- **S3 layout:** `raw/era5/monthly/YYYY-MM/grid_{lat}_{lon}.parquet`
- **Rate limiting:** 1s sleep between calls, no API key needed

**Design rationale:** ERA5 is intentionally coarse — it provides synoptic context (what is the large-scale atmosphere doing?). `boundary_layer_height` encodes marine layer depth; `temp_850hPa` encodes inversion strength. These are primary model inputs, not just background — they explain *why* fog differs across neighborhoods.

---

## Feature Engineering & Modeling Strategy

### Static / Terrain Features

| Feature | Source | Status |
|---------|--------|--------|
| `dist_coast_km` | Computed (manual waypoints) | In S3 |
| `dist_bay_km` | Computed (manual waypoints) | In S3 |
| `elev_m` | Synoptic metadata (feet to m) / DEM | Partial (Synoptic inaccurate, DEM not yet pulled) |
| `coastal_exposure` | Heuristic composite | In S3 (to be replaced by model) |
| Slope, aspect | DEM (SRTM 30m or USGS 3DEP) | Not collected |
| Land cover type | NLCD | Not collected |

### Architecture: Rejected Geo-LSTM-Kriging (2026-03-10)

Han et al. Warsaw results: Kriging RMSE 3.0C, R2 0.58 at mesoscale -- insufficient for neighborhood-scale differentiation. Missing vertical dimension -- SF fog is a vertical phenomenon (marine layer height, inversion) that Geo-LSTM-Kriging doesn't handle. Kriging at neighborhood scale with uneven station density is more likely to hurt than help.

### Stage 1: GBT per variable (build first) [not implemented]

One XGBoost/LightGBM model per target: temperature, humidity, wind speed, wind direction, precipitation.

**Feature vector per station-timestep:**

| Feature Group | Variables | Status |
|---------------|-----------|--------|
| Observation lags | t-1hr, t-3hr, t-6hr per station | Derive from Synoptic |
| Neighbor values | Distance-weighted mean of nearest N stations | Derive from Synoptic |
| ERA5 (interpolated) | `boundary_layer_height` [key], `temp_850hPa` [key], cloud_cover, pressure, wind | Derive from existing ERA5 grid |
| Terrain (static) | Elevation, coastal distance, slope aspect | Partial -- need DEM |
| Land cover (static) | Urban/vegetation/water | Need NLCD |

`boundary_layer_height` = marine layer depth, controls fog penetration. `temp_850hPa` = inversion strength. These are the key features that encode *why* fog differs across neighborhoods. Upgrades ERA5 from background context to primary model input.

### Stage 2: LSTM for temperature & humidity [not implemented]

Wrap LSTM around best GBT feature set for temp/humidity only. These have strong diurnal cycles and sequential memory. Wind and precip are event-driven -- less benefit from temporal modeling.

- Input: 24-hour rolling window of observations + ERA5 features per station
- Terrain features as static context (not part of sequence)
- Requires per-station continuous time series with gap handling
- Need ~6 months clean history per station (have 13 months)

### Stage 3: Spatial interpolation to microclimate zones [not implemented]

Predict at arbitrary lat/lon points across the Bay Area using only ERA5 + terrain features as inputs (no Synoptic observations needed -- the model has learned to predict from spatial/atmospheric features alone). The prediction grid is defined by us at whatever resolution we choose. This produces a continuous weather map.

Cluster on predicted profiles (fog frequency, temp variance, diurnal patterns) -- microclimate zones emerge from learned behavior. Compare against legacy K-means baseline (`cluster_zones.py`).

---

## Synoptic Observation Schema

| Column | Type | Notes |
|--------|------|-------|
| `datetime` | timestamp | UTC |
| `temp_c` | float | Celsius |
| `humidity` | float | Relative humidity % |
| `wind_speed_kph` | float | km/h |
| `wind_dir_deg` | float | Degrees (0-360) |
| `precip_mm` | float | Precipitation, mm |
| `stid` | string | Station ID (e.g. `KSFO`) |
| `name` | string | Station name |
| `lat` | float | Latitude |
| `lon` | float | Longitude |
| `elev_m` | float | **Always meters** (Synoptic: converted from feet) |
| `network` | string | Synoptic network ID |
| `source` | string | `"synoptic"` |

### ERA5 reanalysis (separate schema)

| Column | Type | Notes |
|--------|------|-------|
| `datetime` | timestamp | UTC |
| `temp_2m_c` | float | Surface temperature, °C |
| `humidity_2m` | float | Surface relative humidity % |
| `pressure_hpa` | float | Surface pressure, hPa |
| `wind_speed_10m_kph` | float | km/h |
| `wind_dir_10m_deg` | float | Degrees |
| `precip_mm` | float | Hourly precipitation, mm |
| `cloud_cover_pct` | float | Total cloud cover % |
| `boundary_layer_height_m` | float | PBL height, key for marine layer |
| `temp_850hpa_c` | float | Free-atmosphere temp, proxy for inversion |
| `grid_lat` | float | Grid point latitude |
| `grid_lon` | float | Grid point longitude |

---

## Secrets & Credentials

| Secret | Env Var | Status |
|--------|---------|--------|
| Synoptic API token | `SYNOPTIC_TOKEN` | Active (rotated after exposure incident) |
| AWS credentials | `~/.aws/credentials` | Active (boto3 auto-managed) |

*ERA5 downloads (via Open-Meteo) require no API key.*

**Security notes:**
- All secrets via env vars, never hardcoded
- `.gitignore` excludes `*.env`, `*accessKeys.csv`
- Both Synoptic token and AWS creds were rotated after a past terminal exposure incident

---

## Dependencies

| Package | Purpose |
|---------|---------|
| `pandas` | Data manipulation |
| `boto3` / `botocore` | S3 upload/download |
| `pyarrow` | Parquet serialization |
| `requests` | API calls |
| `python-dotenv` | Load `.env` files |
| `scikit-learn` | K-means clustering |
| `matplotlib` | Elbow plots |
| `dateutil` | Date arithmetic |

---

## Data Gaps & Next Steps

### Stage 1 blockers (fill these first)

| Data | Source | Status |
|------|--------|--------|
| Synoptic chunks 18-22 (65 files) | `download_synoptic.py` | Missing -- run to completion |
| DEM terrain: elevation, slope, aspect | SRTM 30m or USGS 3DEP (free) | Not collected |
| Land cover type (urban/veg/water) | NLCD (free) | Not collected |
| ERA5 to station interpolation | Bilinear interp from existing grid | Write preprocessing step |
| Lag features (t-1hr, t-3hr, t-6hr) | Derive from Synoptic | Preprocessing |
| Neighbor station features | Distance-weighted mean of nearest N | Preprocessing |

### Stage 2 dependencies

| Data | Status |
|------|--------|
| Per-station continuous time series | Restructure from chunked Synoptic |
| Gap handling / imputation | Filter stations with >X% missing |

### Stage 3 dependencies

| Data | Status |
|------|--------|
| Define prediction grid (regular lat/lon mesh) | Choose resolution, generate grid points |
| ERA5 interpolated to prediction grid points | Derive from existing grid |
| DEM + NLCD at prediction grid points | Same sources as Stage 1 |

### Decided

- [x] **Data strategy** -- 10yr ERA5 for ENSO coverage; Synoptic 1yr primary training data (real observations)
- [x] **ML architecture** -- GBT then LSTM then prediction grid interpolation. Not Geo-LSTM-Kriging.
- [x] **ERA5 role** -- Primary model input (boundary_layer_height, temp_850hPa), not just background
- [x] **Predict separately** -- One model per variable (temp, humidity, wind, precip)
- [x] **Open-Meteo dense grid removed** -- Model-interpolated data, not real observations. Would replicate another model's assumptions.

### Modeling decisions (after data gaps filled)

- [ ] Define train/val/test split strategy
- [ ] Train Stage 1 GBT, evaluate feature importance
- [ ] Stage 2: LSTM for temp/humidity
- [ ] Stage 3: predict at prediction grid, cluster into microclimate zones